<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module340/Lab4.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 4 — Four-Qubit H₂ Encoding and Jordan–Wigner Excitations
**Quantum Optimization and Simulation — VQE Laboratory Series**

Use the standard spin-orbital occupation model before reducing to two qubits.

**Suggested use:** 10–15 minute instructor demonstration followed by approximately one hour of independent work.

**Notebook style:** Most code is supplied. Complete the small items marked **YOUR TURN** and answer the reflection questions.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`. When orbital labels are written in the order `q0, q1, ...`, this notebook explicitly notes the convention.

## Learning objectives
- Use the standard four-spin-orbital representation of H₂.
- Prepare the Hartree–Fock occupation state.
- Enumerate valid two-electron configurations.
- See how Jordan–Wigner translates excitation operators into Pauli strings.

In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

SEED = 123
SHOTS = 4096

def run_counts(qc, shots=SHOTS, noise_model=None):
    backend = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=shots, seed_simulator=SEED).result()
    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

## Four spin orbitals

| Qubit | Spin orbital |
|---|---|
| \(q_0\) | bonding, spin up |
| \(q_1\) | bonding, spin down |
| \(q_2\) | antibonding, spin up |
| \(q_3\) | antibonding, spin down |

We use `1 = occupied`, `0 = empty`.

## Part A — Prepare Hartree–Fock

In [ ]:
hf = QuantumCircuit(4)
hf.x(0)
hf.x(1)

display(hf.draw("mpl"))
print(Statevector.from_instruction(hf))

In the logical order \(q_0q_1q_2q_3\), this is `1100`. Qiskit's statevector label is displayed in reverse qubit order.

## Part B — Enumerate all two-electron occupation configurations

In [ ]:
from itertools import product

configurations = [
    bits for bits in product([0,1], repeat=4)
    if sum(bits) == 2
]

print("Number of two-electron configurations:", len(configurations))
for bits in configurations:
    print("".join(map(str, bits)))

**Expected:** \(inom{4}{2}=6\) configurations.

## Part C — Two operators that must not be confused

For the orbital pair \(0\leftrightarrow2\), two related fermionic operators are common.

### 1. Hermitian hopping operator

\[
K=a_2^\dagger a_0+a_0^\dagger a_2.
\]

Jordan–Wigner maps it to

\[
K=\frac12\left(X_0Z_1X_2+Y_0Z_1Y_2\right).
\]

This is the form that appears in a **Hamiltonian hopping term**. Its evolution,

\[
e^{-i\theta K},
\]

is iSWAP-like: starting from one occupation configuration, the transferred
component naturally carries a relative factor of \(i\).

### 2. Anti-Hermitian orbital-excitation generator

For coupled-cluster/UCC state preparation, the generator is instead

\[
A=a_2^\dagger a_0-a_0^\dagger a_2,
\qquad A^\dagger=-A.
\]

The unitary orbital rotation is

\[
U(\theta)=e^{\theta A}.
\]

Equivalently, define the Hermitian operator \(G=iA\), giving

\[
U(\theta)=e^{-i\theta G}.
\]

Its Jordan–Wigner form uses **mixed \(X/Y\) strings**, rather than
\(XZX+YZY\). For real orbitals and a real starting state, this is the
conventional Givens-style rotation that produces a real superposition.

> Therefore, `IXZX + IYZY` is correct for the Hermitian hopping sum, but it
> should not be presented as the UCC single-excitation generator.


In [ ]:
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

mapper = JordanWignerMapper()

# (1) Hermitian hopping:
#     K = a_2^† a_0 + a_0^† a_2
hopping_sum = FermionicOp(
    {
        "+_2 -_0": 1.0,
        "+_0 -_2": 1.0,
    },
    num_spin_orbitals=4,
)

jw_hopping_sum = mapper.map(hopping_sum)

print("Hermitian hopping sum K:")
print(jw_hopping_sum)

# Expected Qiskit labels (left to right: q3 q2 q1 q0):
# 0.5 * IXZX + 0.5 * IYZY


In [ ]:
# (2) Anti-Hermitian UCC single excitation:
#     A = a_2^† a_0 - a_0^† a_2
ucc_single = FermionicOp(
    {
        "+_2 -_0": 1.0,
        "+_0 -_2": -1.0,
    },
    num_spin_orbitals=4,
)

jw_ucc_single = mapper.map(ucc_single)

print("Anti-Hermitian excitation generator A:")
print(jw_ucc_single)

# It has imaginary coefficients on mixed X/Y strings, equivalent,
# up to ordering/sign convention, to:
#
# A = (i/2)(X_0 Z_1 Y_2 - Y_0 Z_1 X_2)

# Convert to the Hermitian generator G = iA so that U(theta)=exp(-i theta G).
jw_givens_generator = 1j * jw_ucc_single

print("\nHermitian G = iA:")
print(jw_givens_generator)


### Comment on \(R_{XX}(\theta)R_{YY}(\theta)\)

\[
R_{XX}(\theta)R_{YY}(\theta)
=e^{-i\theta(XX+YY)/2}
\]

implements evolution generated by the **Hermitian hopping sum**. It preserves
particle number, but its natural transition amplitude is imaginary.

That does **not** make the quantum state invalid; quantum states may be complex.
However, it is not the same as the conventional real orbital rotation generated
by \(T-T^\dagger\).

For two adjacent reduced-encoding qubits, a phase-adjusted \(XX+YY\) gate can
implement the real rotation. In Qiskit:

```python
from qiskit.circuit.library import XXPlusYYGate
qc.append(XXPlusYYGate(2*theta, beta=np.pi/2), [0, 1])
```

This gives, up to a convention-dependent sign,

\[
|01\rangle\mapsto
\cos\theta\,|01\rangle+\sin\theta\,|10\rangle.
\]


## Part D — Double excitation

In [ ]:
double_generator = FermionicOp(
    {
        "+_2 +_3 -_1 -_0": 1.0,
        "+_0 +_1 -_3 -_2": -1.0,
    },
    num_spin_orbitals=4,
)

jw_double = JordanWignerMapper().map(double_generator)
print(jw_double)

### YOUR TURN
Count the Pauli strings in `jw_double` and identify how many contain one \(Y\) and how many contain three \(Y\)'s.

In [ ]:
labels = jw_double.paulis.to_labels()
print(labels)

# TODO: calculate these two counts.
one_y = 0
three_y = 0

print("one Y:", one_y)
print("three Y:", three_y)

<details>
<summary><b>Instructor solution / suggested answer</b></summary>


    A conventional Jordan–Wigner double-excitation generator produces eight nonzero Pauli strings: four with one \(Y\) and four with three \(Y\)'s, up to ordering/sign conventions.

    Suggested code:
    ```python
    one_y = sum(label.count("Y") == 1 for label in labels)
    three_y = sum(label.count("Y") == 3 for label in labels)
    ```

</details>